# Notebook 0: Prepare Grouped Dataset For Training

This notebook prepares a user-provided class-root dataset for Notebook 2. The public default does not require access to the private research dataset.

Class-root datasets shipped in the repo: data/class_root_dataset/tomato_fruit, data/class_root_dataset/grape_fruit and data/class_root_dataset/grape_leaf.

The audit goal is not just "clean" data but data that can be trusted for adapter performance:
- keep exact-duplicate and near-duplicate families together.
- do not split capture/session/source families; prevent source leakage.
- manage real hard negatives and the OOD pool separately from the classification classes.
- materialize because class support and source distribution are trustworthy, not only because the audit report is clean.

Public usage:
1. Leave `DATASET_RELEASE_TAG` empty.
2. Upload or mount your class-root dataset and set `REPO_DATASET_ROOT` or `DATASET_ROOT`.
3. Run the audit and materialization cells in order.
4. For a zero-data smoke run, use Notebook 2's `public_sample` default instead.

The maintainer local audit flow runs when `DATASET_RELEASE_TAG` is left empty.


## Quick Flow

With a release tag set, the notebook downloads the immutable GitHub Release, verifies the manifest and hashes, and materializes the `continual/`, `val/`, `test/`, `ood/`, `oe/` layout that Notebook 2 consumes under the cache root.

- For the release flow, use only the fixed repository/tag/target/cache parameters.
- The notebook does not ask for a write token and does not commit or push datasets.
- A token or network failure is reported as `DATASET_RELEASE_ACCESS_BLOCKER`; a hash/manifest mismatch is a separate integrity error.
- The local class-root audit flow runs only when the release tag is empty.


## Immutable GitHub Release Selection

The private immutable Release path below is maintainer-only. Public users should leave the tag empty and provide their own data.

### Usage

1. Set `DATASET_RELEASE_REPOSITORY` and the immutable `DATASET_RELEASE_TAG`.
2. Choose the `<crop>__<part>` target with `DATASET_RELEASE_TARGET`.
3. Set the cache root and add the read-only token as a Colab secret.
4. Run the access check and materialization cells in order.

### Example

```python
DATASET_RELEASE_REPOSITORY = "EfeErim/aads-open-world-plant-disease"
DATASET_RELEASE_TAG = "aads-dataset-v1.0.0"
DATASET_RELEASE_TARGET = "tomato__leaf"
```

This flow does not depend on Drive or on dataset Git commits; release assets are verified against the manifest and SHA-256.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token
    os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/aads-open-world-plant-disease.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root

    if not CLONE_TARGET.exists():
        CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
        completed = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.returncode != 0:
            raise RuntimeError(
                'Notebook 0 repo bootstrap failed. Set AADS_REPO_ROOT to an existing checkout '
                'or provide the read-only AADS_GITHUB_RELEASE_READ_TOKEN Colab secret.'
            )

    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell02_bootstrap.py', globals())

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell03_runtime_setup.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parameters ---
    # Edit this cell first, then run the cells below in order.

    # --- QUICK USE: keep or change these four values for the read-only release ---
    DATASET_RELEASE_REPOSITORY = "EfeErim/aads-open-world-plant-disease"
    DATASET_RELEASE_TAG = ""
    DATASET_RELEASE_TARGET = "tomato__leaf"
    DATASET_RELEASE_CACHE_ROOT = ".runtime_tmp/dataset_release_cache"

    # If DATASET_RELEASE_TAG is empty, the maintainer audit/materialize path below is used.
    # REPO_DATASET_ROOT: a repo-relative or absolute path.
    REPO_DATASET_ROOT = "data/class_root_dataset"  # replace with your own class-root path

    # REPO_DATASET_NAME: dataset name when REPO_DATASET_ROOT is a parent folder; leave empty for a direct dataset path.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: optional override. If empty, the selection above is used.
    DATASET_ROOT = ""

    # IMPORT_FROM_DRIVE: True asks for Drive options when a Drive path is set; the default flow stays inside the repo.
    IMPORT_FROM_DRIVE = False

    # DRIVE_DATASET_PATH: optional Drive dataset parent path. If empty, the repo data folder is used.
    DRIVE_DATASET_PATH = ""

    # DRIVE_DATASET_NAME: folder to copy. If empty, the options are listed and the notebook asks.
    DRIVE_DATASET_NAME = ""

    # CROP_NAME / PART_NAME: if empty, the notebook asks.
    CROP_NAME = ""
    PART_NAME = ""

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "dataset_name": None,
        "dataset_source": None,
        "selected_ood_dataset_name": None,
        "resolved_ood_root": None,
        "prep_materialization_result": None,
        "access_report": None,
        "dataset_release_report": None,
    }
    print(
        f"[PARAMS] run_id={run_id} crop={CROP_NAME or '(ask)'} part={PART_NAME or '(ask)'} "
        f"repo_dataset_root={REPO_DATASET_ROOT} repo_dataset_name={REPO_DATASET_NAME or '(ask)'} "
        f"dataset_root_override={DATASET_ROOT or '(none)'}"
    )
    print(
        f"[DRIVE] import_from_drive={IMPORT_FROM_DRIVE} drive_path={DRIVE_DATASET_PATH or '<none>'} "
        f"drive_name={DRIVE_DATASET_NAME or '<ask>'}"
    )
    print(
        f"[OOD] repo_root={OOD_DATASET_ROOT} repo_name={OOD_DATASET_NAME or '<none>'} direct_root={OOD_ROOT or '<none>'}"
    )
    print(
        f"[DATASET RELEASE] repository={DATASET_RELEASE_REPOSITORY} tag={DATASET_RELEASE_TAG or '<local-audit>'} "
        f"target={DATASET_RELEASE_TARGET or '<none>'} cache_root={DATASET_RELEASE_CACHE_ROOT}"
    )
    print(
        "[NEXT] Access check -> audit -> review guided/00_start_here.md -> "
        f"materialize={MATERIALIZE_AFTER_REVIEW}. Runtime output: {PREPARED_RUNTIME_ROOT}"
    )
    PREPARE_DATASET_FROM_REPORTS = PREPARE_DATASET_FROM_REPORTS and not bool(DATASET_RELEASE_TAG)
    MATERIALIZE_AFTER_REVIEW = MATERIALIZE_AFTER_REVIEW and not bool(DATASET_RELEASE_TAG)


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3a: Google Drive Dataset Import (Optional)"):
    from src.data.dataset_layout import resolve_dataset_directory_from_parent
    from src.colab.repo_bootstrap import mount_drive_if_available
    import shutil
    
    def import_dataset_from_drive(
        source_path: str | Path,
        destination_path: str | Path,
        dataset_name: str,
        overwrite: bool = False
    ) -> bool:
        """Copy a dataset from Drive into a local folder."""
        source_path = Path(source_path)
        destination_path = Path(destination_path).expanduser()
        destination_path.mkdir(parents=True, exist_ok=True)
        
        target = destination_path / dataset_name
        
        if target.exists():
            if not overwrite:
                print(f"[DRIVE] Target folder already exists: {target}")
                return False
            shutil.rmtree(target)
            print(f"[DRIVE] Removed existing folder: {target}")
        
        try:
            print(f"[DRIVE] Copy started: {source_path} -> {target}")
            shutil.copytree(source_path, target, dirs_exist_ok=False)
            print(f"[DRIVE] Copied: {target}")
            return True
        except Exception as e:
            print(f"[DRIVE] Copy failed: {e}")
            return False
    
    def _drive_destination_parent() -> Path:
        explicit_root = str(DATASET_ROOT).strip()
        if explicit_root:
            root_path = Path(explicit_root).expanduser()
            return root_path.resolve() if root_path.is_absolute() else (ROOT / root_path).resolve()
        return ROOT / "data" / "imported_from_drive"
    
    # Optional dataset import from Google Drive (driven by the parameters)
    if IMPORT_FROM_DRIVE and str(DRIVE_DATASET_PATH).strip():
        print("[DRIVE] Looking for the dataset in Google Drive...")
        mount_drive_if_available()
        
        drive_path = Path(DRIVE_DATASET_PATH).expanduser()
        if not drive_path.exists():
            raise RuntimeError(f"Drive path not found: {drive_path}")
        dataset_name, source, available_datasets = resolve_dataset_directory_from_parent(
            dataset_parent=drive_path,
            requested_name=DRIVE_DATASET_NAME,
            prompt_label="Drive dataset",
        )
        print(f"[DRIVE] available datasets={available_datasets}")
        dest_parent = _drive_destination_parent()
        if import_dataset_from_drive(source, dest_parent, dataset_name):
            DATASET_ROOT = str(dest_parent / dataset_name)
            print(f"[DRIVE] DATASET_ROOT updated: {DATASET_ROOT}")
        else:
            existing_target = dest_parent / dataset_name
            if existing_target.is_dir():
                DATASET_ROOT = str(existing_target)
                print(f"[DRIVE] Using the existing copy, DATASET_ROOT updated: {DATASET_ROOT}")
    else:
        print("[DRIVE] IMPORT_FROM_DRIVE=False or DRIVE_DATASET_PATH is empty. Skipping Drive import.")

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell06_access_check.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_dataset_audit

    if DATASET_RELEASE_TAG:
        STATE["validated"] = True
        STATE["audit_summary"] = {"runtime_ready": True, "source": "github_release"}
        print("[DATASET RELEASE] Local class-root audit skipped; immutable release verification is authoritative.")
    else:
        STATE, CROP_NAME, PART_NAME = run_dataset_audit(
            ROOT=ROOT,
            STATE=STATE,
            TELEMETRY=TELEMETRY,
            DEVICE=DEVICE,
            EMBEDDING_BATCH_SIZE=EMBEDDING_BATCH_SIZE,
            NEIGHBORS=NEIGHBORS,
            REPO_DATASET_ROOT=REPO_DATASET_ROOT,
            REPO_DATASET_NAME=REPO_DATASET_NAME,
            DATASET_ROOT=DATASET_ROOT,
            CROP_NAME=CROP_NAME,
            PART_NAME=PART_NAME,
            PREP_ARTIFACT_ROOT=PREP_ARTIFACT_ROOT,
            PREP_DINOV3_MODEL_ID=PREP_DINOV3_MODEL_ID,
            PREP_BIOCLIP_MODEL_ID=PREP_BIOCLIP_MODEL_ID,
            UNDER_MIN_EVAL_POLICY=UNDER_MIN_EVAL_POLICY,
            IMPORT_FROM_DRIVE=IMPORT_FROM_DRIVE,
            DRIVE_DATASET_PATH=DRIVE_DATASET_PATH,
            INTERACTIVE_AUDIT_REVIEW=INTERACTIVE_AUDIT_REVIEW,
            MAX_INTERACTIVE_REVIEW_ITEMS=MAX_INTERACTIVE_REVIEW_ITEMS,
        )


In [ ]:
if "TELEMETRY" not in globals() or "STATE" not in globals():
    raise RuntimeError("Runtime setup is not ready. Run the bootstrap and Cell 3 runtime setup cells, then the Cell 4 Dataset Audit cells, in order.")

with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from src.data.grouped_runtime_dataset_preparation import build_human_review_packet, build_prepared_dataset_key, format_human_review_packet
    from scripts.prepare_materialization_dataset import prepare_class_root_for_materialization
    from src.shared.json_utils import write_json

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Run the dataset audit cell first.")

    summary = STATE["audit_summary"]
    if not PREPARE_DATASET_FROM_REPORTS:
        print("[PREP] PREPARE_DATASET_FROM_REPORTS=False. Audit reports left unused.")
    else:
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        prepared_class_root_parent = Path(PREPARED_CLASS_ROOT).expanduser()
        if not prepared_class_root_parent.is_absolute():
            prepared_class_root_parent = (ROOT / prepared_class_root_parent).resolve()
        prepared_class_root = prepared_class_root_parent / dataset_key
        prepared_artifact_root = STATE["artifact_root"].parent / f"{STATE['artifact_root'].name}_prepared"
        prep_counts = dict(summary.get("summary", {}))
        print("[PREP] Report-based preparation summary:")
        print(
            f"  dataset_key={dataset_key} total_images={prep_counts.get('total_images', 0)} "
            f"cross_class_conflicts={prep_counts.get('cross_class_conflicts', 0)} "
            f"same_class_high_risk_clusters={prep_counts.get('same_class_high_risk_clusters', 0)}"
        )
        print(f"  source_dataset_root={STATE['dataset_root']}")
        print(f"  source_artifact_root={STATE['artifact_root']}")
        print(f"  prepared_class_root={prepared_class_root}")
        print(f"  prepared_artifact_root={prepared_artifact_root}")
        print(f"  cleanup_seed={CLEANUP_SEED}")
        prep_result = prepare_class_root_for_materialization(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            audit_artifact_root=STATE["artifact_root"],
            prepared_class_root=prepared_class_root,
            prepared_artifact_root=prepared_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            dino_model_id=PREP_DINOV3_MODEL_ID,
            bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
            neighbors=NEIGHBORS,
            cleanup_seed=CLEANUP_SEED,
            quarantine_cross_class_conflicts=True,
            under_min_eval_policy=UNDER_MIN_EVAL_POLICY,
            materialization_strategy="auto",
            progress_fn=lambda message: print(f"[PREP] {message}"),
        )
        STATE["prep_materialization_result"] = prep_result
        STATE["dataset_root"] = Path(prep_result["prepared_class_root"])
        STATE["dataset_source"] = "prepared_class_root"
        STATE["artifact_root"] = Path(prep_result["prepared_artifact_root"])
        STATE["audit_summary"] = prep_result["rerun_summary"]
        summary = STATE["audit_summary"]
        review_packet = build_human_review_packet(
            summary,
            artifact_root=STATE["artifact_root"],
            max_review_items=MAX_INTERACTIVE_REVIEW_ITEMS,
        )
        STATE["human_review_packet"] = review_packet
        STATE["human_review_approved"] = not bool(review_packet.get("pause_recommended"))
        write_json(STATE["artifact_root"] / "human_review_packet.json", review_packet, ensure_ascii=False)
        print(
            f"[PREP] prepared_runtime_ready={prep_result.get('prepared_runtime_ready')} "
            f"dataset_key={prep_result.get('dataset_key')} prepared_class_root={STATE['dataset_root']}"
        )
        print(f"[PREP] Post-preparation artifact_root={STATE['artifact_root']}")
        print(format_human_review_packet(review_packet))
        if INTERACTIVE_AUDIT_REVIEW and review_packet.get("pause_recommended"):
            print("[PREP] Prepared audit gate: Enter applies the safe default.")
            if not _prompt_yes_no("Continue with the prepared dataset?", True):
                STATE["human_review_stop_requested"] = True
                STATE["human_review_approved"] = False
                raise RuntimeError("Stopped by human review. Review the artifacts and re-run the cell.")
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        else:
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_prepared",
                "dataset_root": str(STATE["dataset_root"]),
                "dataset_source": str(STATE.get("dataset_source") or "prepared_class_root"),
                "artifact_root": str(STATE["artifact_root"]),
                "runtime_ready": bool(summary.get("runtime_ready")),
            }
        )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_materialize_runtime_dataset

    run_materialize_runtime_dataset(
        ROOT=ROOT,
        STATE=STATE,
        TELEMETRY=TELEMETRY,
        CROP_NAME=CROP_NAME,
        PART_NAME=PART_NAME,
        OOD_ROOT=OOD_ROOT,
        OOD_DATASET_NAME=OOD_DATASET_NAME,
        OOD_DATASET_ROOT=OOD_DATASET_ROOT,
        ASK_FOR_OOD_ROOT=ASK_FOR_OOD_ROOT,
        PREPARED_RUNTIME_ROOT=PREPARED_RUNTIME_ROOT,
        MATERIALIZE_AFTER_REVIEW=MATERIALIZE_AFTER_REVIEW,
        DATASET_RELEASE_REPOSITORY=DATASET_RELEASE_REPOSITORY,
        DATASET_RELEASE_TAG=DATASET_RELEASE_TAG,
        DATASET_RELEASE_TARGET=DATASET_RELEASE_TARGET,
        DATASET_RELEASE_CACHE_ROOT=DATASET_RELEASE_CACHE_ROOT,
        REPO_NOTEBOOK_OUTPUT_PATH=REPO_NOTEBOOK_OUTPUT_PATH,
        REPO_RUN_DIR=REPO_RUN_DIR,
        REPO_RUN_EXPORTS=REPO_RUN_EXPORTS,
        INTERACTIVE_AUDIT_REVIEW=INTERACTIVE_AUDIT_REVIEW,
    )
